# Surrogate Models in Two Acts: Physics Emulation and xAI

This Colab-ready notebook uses only packages included in a standard Google Colab runtime. It first replaces a deliberately slower 1-D heat-diffusion solver with a Gaussian-process (Kriging-style) emulator. It then treats that trained emulator as a black box and explains it with a local weighted-linear surrogate and a global distilled decision tree.

**Expected runtime:** about 10–20 seconds on a standard Colab CPU. No `pip install` cells are needed.

## Setup
Run this cell first.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree

SEED = 42
RNG = np.random.default_rng(SEED)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

# Part 1 — A physical-model surrogate

The reference model is a small finite-difference heat solver. Its two inputs are heat-source strength and thermal diffusivity, and its output is peak temperature after a fixed simulated duration. It is intentionally simple enough for a workshop while retaining the repeated-time-stepping cost that motivates an emulator.

In [ ]:
# Cell 1: run the slower reference physics model for a small design of experiments.
# Increase N_STEPS to make the reference model more expensive; do not change dt.
N_X = 80
N_STEPS = 3000
x = np.linspace(0.0, 1.0, N_X)
dx = x[1] - x[0]
dt = 0.40 * dx**2 / 0.25  # stable at the largest diffusivity used below

PARAMETER_BOUNDS = np.array([[0.50, 2.50], [0.05, 0.25]])
feature_names = ["Heat-source strength", "Thermal diffusivity"]

def physics_model(source_strength, diffusivity):
    """Return peak temperature from a 1-D heat-diffusion simulation."""
    temperature = np.zeros(N_X)
    source = source_strength * np.exp(-0.5 * ((x - 0.50) / 0.11) ** 2)
    diffusion_number = diffusivity * dt / dx**2

    for _ in range(N_STEPS):
        next_temperature = temperature.copy()
        next_temperature[1:-1] = (
            temperature[1:-1]
            + diffusion_number * (temperature[2:] - 2 * temperature[1:-1] + temperature[:-2])
            + dt * source[1:-1]
        )
        temperature = next_temperature

    return temperature.max()

n_train = 64
X_train = RNG.uniform(PARAMETER_BOUNDS[:, 0], PARAMETER_BOUNDS[:, 1], size=(n_train, 2))

start = time.perf_counter()
y_train = np.array([physics_model(*inputs) for inputs in X_train])
reference_training_seconds = time.perf_counter() - start

print(f"Generated {n_train} reference simulations in {reference_training_seconds:.2f} seconds.")
print(f"Peak-temperature range: {y_train.min():.3f} to {y_train.max():.3f}")

In [ ]:
# Cell 2: fit a Gaussian-process / Kriging-style emulator to the reference runs.
# optimizer=None keeps the workshop version fast and reproducible.
surrogate = make_pipeline(
    StandardScaler(),
    GaussianProcessRegressor(
        kernel=ConstantKernel(1.0) * RBF(length_scale=1.5),
        alpha=1e-8,
        normalize_y=True,
        optimizer=None,
        random_state=SEED,
    ),
)

start = time.perf_counter()
surrogate.fit(X_train, y_train)
surrogate_fit_seconds = time.perf_counter() - start

print(f"Fitted the Kriging-style surrogate in {surrogate_fit_seconds:.3f} seconds.")

In [ ]:
# Cell 3: compare reference-solver fidelity and prediction time over several parameter settings.
source_values = np.linspace(0.55, 2.45, 16)
diffusivities = np.array([0.07, 0.15, 0.23])
scenario_labels = ["low diffusivity", "middle diffusivity", "high diffusivity"]

X_eval = np.array(
    [[source_strength, diffusivity]
     for diffusivity in diffusivities
     for source_strength in source_values],
    dtype=float,
)

start = time.perf_counter()
y_physics = np.array([physics_model(*inputs) for inputs in X_eval])
physics_seconds = time.perf_counter() - start

start = time.perf_counter()
y_surrogate = surrogate.predict(X_eval)
surrogate_seconds = time.perf_counter() - start

mae = mean_absolute_error(y_physics, y_surrogate)
r2 = r2_score(y_physics, y_surrogate)
speedup = physics_seconds / max(surrogate_seconds, np.finfo(float).eps)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.7), gridspec_kw={"width_ratios": [1.45, 1, 0.8]})

# Response curves: same color means same diffusivity; line style distinguishes model.
for i, diffusivity in enumerate(diffusivities):
    selection = slice(i * len(source_values), (i + 1) * len(source_values))
    axes[0].plot(source_values, y_physics[selection], color=f"C{i}", linewidth=2.5)
    axes[0].plot(source_values, y_surrogate[selection], color=f"C{i}", linestyle="--", linewidth=2.5)

color_handles = [
    Line2D([0], [0], color=f"C{i}", linewidth=2.5, label=f"{label}: k = {diffusivity:.2f}")
    for i, (label, diffusivity) in enumerate(zip(scenario_labels, diffusivities))
]
style_handles = [
    Line2D([0], [0], color="black", linewidth=2.5, label="physics solver"),
    Line2D([0], [0], color="black", linestyle="--", linewidth=2.5, label="Kriging surrogate"),
]
axes[0].legend(handles=color_handles + style_handles, fontsize=7.5, ncol=2)
axes[0].set(title="Response curves across parameter settings", xlabel="Heat-source strength", ylabel="Peak temperature")

# Fidelity: the physics solver is the reference for this toy experiment.
limits = [min(y_physics.min(), y_surrogate.min()), max(y_physics.max(), y_surrogate.max())]
axes[1].scatter(y_physics, y_surrogate, color="#4E81BD", s=34, alpha=0.85)
axes[1].plot(limits, limits, "k--", linewidth=1)
axes[1].set(xlim=limits, ylim=limits, xlabel="Physics-model peak temperature", ylabel="Surrogate peak temperature", title=f"Fidelity\nMAE = {mae:.2e}, R2 = {r2:.5f}")

# Prediction cost only: creating the original reference data remains an upfront cost.
timing_values = [physics_seconds, surrogate_seconds]
bars = axes[2].bar(["Physics\nsolver", "Kriging\nsurrogate"], timing_values, color=["#C7702C", "#49885B"])
axes[2].set_yscale("log")
axes[2].set(ylabel="Seconds for all evaluation cases (log scale)", title=f"Prediction cost\n{speedup:,.0f}x speedup")
for bar, value in zip(bars, timing_values):
    axes[2].text(bar.get_x() + bar.get_width() / 2, value * 1.3, f"{value:.3g} s", ha="center", fontsize=9)

fig.suptitle("Reference Physics Model vs. Gaussian-Process Surrogate", y=1.03)
plt.tight_layout()
plt.show()

print(f"Evaluated {len(X_eval)} parameter combinations.")
print(f"MAE: {mae:.3e} | R2: {r2:.6f}")
print(f"Physics time: {physics_seconds:.3f} s | Surrogate time: {surrogate_seconds:.5f} s")

# Part 2 — Explaining a black-box surrogate

We now treat the trained Gaussian-process emulator as the black box. The local model answers **why this one emulator prediction?** The tree answers **what simple rules approximate the emulator across this input domain?** Neither establishes the causal physics of heat transfer.

In [ ]:
# Create a representative ensemble of cheap black-box predictions, then distill it into a shallow tree.
# This is a global surrogate of the trained emulator, not of the physical system directly.
black_box = surrogate
distill_rng = np.random.default_rng(SEED + 1)
X_distill = distill_rng.uniform(PARAMETER_BOUNDS[:, 0], PARAMETER_BOUNDS[:, 1], size=(1500, 2))
teacher_distill = black_box.predict(X_distill)

global_tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=20, random_state=SEED)
global_tree.fit(X_distill, teacher_distill)

teacher_eval = black_box.predict(X_eval)
tree_eval = global_tree.predict(X_eval)
global_fidelity = r2_score(teacher_eval, tree_eval)

print(f"Global tree fidelity to the black-box emulator (R2): {global_fidelity:.3f}")

In [ ]:
# Build a LIME-style local weighted-linear surrogate around one plausible input case.
# Inputs are perturbed and clipped to the physical-model training domain.
example_id = len(X_eval) // 2
x0 = X_eval[example_id]
black_box_prediction = black_box.predict(x0.reshape(1, -1))[0]

feature_span = PARAMETER_BOUNDS[:, 1] - PARAMETER_BOUNDS[:, 0]
local_rng = np.random.default_rng(SEED + 2)
local_X = x0 + local_rng.normal(loc=0.0, scale=0.08 * feature_span, size=(1200, 2))
local_X = np.clip(local_X, PARAMETER_BOUNDS[:, 0], PARAMETER_BOUNDS[:, 1])
local_y = black_box.predict(local_X)

# Scale around x0 so a coefficient represents the effect of a one-range local move.
local_X_scaled = (local_X - x0) / feature_span
distance = np.linalg.norm(local_X_scaled, axis=1)
weights = np.exp(-(distance**2) / (0.65**2))

local_model = Ridge(alpha=0.01)
local_model.fit(local_X_scaled, local_y, sample_weight=weights)
local_prediction = local_model.predict(np.zeros((1, 2)))[0]
local_fidelity = r2_score(local_y, local_model.predict(local_X_scaled), sample_weight=weights)

print("Case being explained:")
print(dict(zip(feature_names, np.round(x0, 3))))
print(f"Black-box prediction: {black_box_prediction:.4f}")
print(f"Local-surrogate prediction: {local_prediction:.4f}")
print(f"Weighted local fidelity (R2): {local_fidelity:.3f}")

In [ ]:
# Visualize the global distilled tree and the local LIME-style explanation.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Global surrogate fidelity to the black-box emulator.
tree_limits = [min(teacher_eval.min(), tree_eval.min()), max(teacher_eval.max(), tree_eval.max())]
axes[0].scatter(teacher_eval, tree_eval, alpha=0.7, color="#4E81BD")
axes[0].plot(tree_limits, tree_limits, "k--", linewidth=1)
axes[0].set(xlim=tree_limits, ylim=tree_limits, xlabel="Black-box emulator prediction", ylabel="Distilled-tree prediction", title=f"Global surrogate fidelity\nR2 = {global_fidelity:.3f}")

# Global explanation.
plot_tree(global_tree, feature_names=feature_names, filled=True, rounded=True, impurity=False, fontsize=7, ax=axes[1])
axes[1].set_title("Global distilled-tree surrogate")

# Local explanation. Positive coefficients raise the emulator's local prediction.
order = np.argsort(local_model.coef_)
ordered_names = np.array(feature_names)[order]
ordered_coefficients = local_model.coef_[order]
bar_colors = np.where(ordered_coefficients >= 0, "#49885B", "#C7702C")
axes[2].barh(ordered_names, ordered_coefficients, color=bar_colors)
axes[2].axvline(0, color="black", linewidth=0.8)
axes[2].set(xlabel="Local weighted-linear coefficient", title=f"Why this emulator prediction?\nLocal R2 = {local_fidelity:.3f}")

plt.tight_layout()
plt.show()

## Interpretation guardrails

- The first comparison checks whether the Gaussian-process surrogate matches the **reference solver** inside the sampled input domain. It does not validate the toy solver against real measurements.
- The shallow tree is a global surrogate of the **trained emulator**. Its R2 is therefore fidelity to that emulator, not a claim about the physical system.
- The weighted-linear explanation is local to the selected case and to the chosen perturbation neighborhood. The clipping step keeps perturbations within the trained physical-model domain.
- For a quick workshop run, lower `N_STEPS` to `1500` or `n_train` to `40`. For a more demanding demonstration, increase them and observe how the physics-solver cost grows while emulator prediction remains cheap.